# Dia 3 — RAG com FAISS: o agente que consulta conhecimento externo

Nos dias anteriores o agente sabia apenas **ler e enviar e-mails**.  
Hoje ele ganha acesso a uma **base de conhecimento** — um catálogo de produtos com preços.

O conceito que vamos usar chama-se **RAG (Retrieval-Augmented Generation)**:  
em vez de colocar tudo no prompt, o agente *busca* apenas o que é relevante no momento.

| Parte | Tema |
|---|---|
| Setup | Conexões, credenciais, login no e-mail |
| A | Criar o índice FAISS com o catálogo de produtos |
| B | Tool `search_products` que consulta o índice |
| C | Agente que busca produtos e envia o resultado por e-mail |

---

## Setup

In [1]:
# Célula 1 — Instalar dependências
# faiss-cpu  → índice vetorial local (sem conta em nenhum serviço externo)
# sentence-transformers → gera embeddings localmente, também sem API externa
!pip install -q langchain-anthropic langchain langgraph faiss-cpu sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\mnsmferr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Preencha as credenciais com os dados fornecidos pelo professor. Neste dia usamos os mesmos servidores dos dias anteriores, mais `faiss-cpu` e `sentence-transformers` para o índice vetorial local.

Verificamos o proxy e configuramos o LLM — idêntico aos dias anteriores.

Fazemos login no servidor de e-mail para poder enviar os resultados da busca ao final do notebook.

In [2]:
# Célula 2 — Credenciais
PROXY_URL      = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN    = "xpto_aluno-01"

EMAIL_URL      = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN    = "aluno-01"
EMAIL_PASSWORD = "1234"

print("Credenciais carregadas.")

Credenciais carregadas.


In [3]:
# Célula 3 — Testar proxy LLM
import requests
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

r = requests.get(f"{PROXY_URL}/health")
print("Proxy:", r.status_code, r.json())

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=512,
)
resp = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print("LLM:", resp.content)

C:\Users\mnsmferr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Proxy: 200 {'status': 'ok', 'anthropic_key_configured': True, 'alunos_registrados': 10}
LLM: Conexão ok!


Com o catálogo em memória, geramos um **embedding** para cada produto e construímos o índice FAISS. Um embedding é um vetor numérico que representa o significado semântico do texto — produtos similares ficam próximos nesse espaço vetorial.

Testamos a busca **diretamente no índice** — sem agente — para entender o mecanismo antes de encapsulá-lo em uma tool. Observe os scores: quanto maior, mais relevante o produto para a query.

In [4]:
# Célula 4 — Login no servidor de e-mail
login = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_PASSWORD,
})
assert login.status_code == 200, f"Login falhou: {login.json()}"
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Login OK →", login.json())

Login OK → {'token': 'aluno-01', 'name': 'Aluno 1', 'email': 'aluno01@curso.ia'}


---
## Parte A — Criando o índice FAISS

### O que é RAG?

Um LLM tem um limite de contexto e não conhece dados específicos do seu negócio.  
O RAG resolve isso em três passos:

1. **Indexar** — transformar documentos em vetores numéricos (embeddings) e armazená-los
2. **Buscar** — dado uma pergunta, encontrar os documentos mais similares
3. **Gerar** — passar esses documentos como contexto para o LLM responder

Hoje usamos **FAISS** (Facebook AI Similarity Search) como índice vetorial — roda 100% local, sem API externa.

Adicionamos `send_email` — a mesma tool dos dias anteriores — para que o agente possa enviar os resultados da busca por e-mail.

In [5]:
# Célula 5 — Catálogo de produtos (nossa "base de conhecimento")

catalogo = [
    # Vestuário
    {"nome": "Calça jeans masculina",      "categoria": "vestuario", "preco": 129.90},
    {"nome": "Calça social feminina",       "categoria": "vestuario", "preco": 149.90},
    {"nome": "Blusa de algodão feminina",   "categoria": "vestuario", "preco": 59.90},
    {"nome": "Blusa polo masculina",        "categoria": "vestuario", "preco": 79.90},
    {"nome": "Moletom unissex com capuz",   "categoria": "vestuario", "preco": 189.90},
    {"nome": "Saia midi floral",            "categoria": "vestuario", "preco": 99.90},
    {"nome": "Vestido casual listrado",     "categoria": "vestuario", "preco": 119.90},
    {"nome": "Jaqueta corta-vento",         "categoria": "vestuario", "preco": 249.90},
    # Outros
    {"nome": "Brinquedo educativo blocos",  "categoria": "brinquedo", "preco": 89.90},
    {"nome": "Carrinho de controle remoto", "categoria": "brinquedo", "preco": 159.90},
    {"nome": "Jogo de louça 12 peças",      "categoria": "loucas",    "preco": 219.90},
    {"nome": "Frigideira antiaderente",     "categoria": "loucas",    "preco": 79.90},
    {"nome": "Quadro decorativo abstrato",  "categoria": "decoracao", "preco": 139.90},
    {"nome": "Espelho redondo 60cm",        "categoria": "decoracao", "preco": 179.90},
    {"nome": "Smartphone 128GB",            "categoria": "eletronico","preco": 1299.90},
    {"nome": "Fone de ouvido bluetooth",    "categoria": "eletronico","preco": 199.90},
]

print(f"Catálogo com {len(catalogo)} produtos carregado.")
for p in catalogo:
    print(f"  [{p['categoria']:12}] {p['nome']:<35} R$ {p['preco']:.2f}")

Catálogo com 16 produtos carregado.
  [vestuario   ] Calça jeans masculina               R$ 129.90
  [vestuario   ] Calça social feminina               R$ 149.90
  [vestuario   ] Blusa de algodão feminina           R$ 59.90
  [vestuario   ] Blusa polo masculina                R$ 79.90
  [vestuario   ] Moletom unissex com capuz           R$ 189.90
  [vestuario   ] Saia midi floral                    R$ 99.90
  [vestuario   ] Vestido casual listrado             R$ 119.90
  [vestuario   ] Jaqueta corta-vento                 R$ 249.90
  [brinquedo   ] Brinquedo educativo blocos          R$ 89.90
  [brinquedo   ] Carrinho de controle remoto         R$ 159.90
  [loucas      ] Jogo de louça 12 peças              R$ 219.90
  [loucas      ] Frigideira antiaderente             R$ 79.90
  [decoracao   ] Quadro decorativo abstrato          R$ 139.90
  [decoracao   ] Espelho redondo 60cm                R$ 179.90
  [eletronico  ] Smartphone 128GB                    R$ 1299.90
  [eletronico  ] Fone d

In [6]:
# Célula 6 — Gerar embeddings e construir o índice FAISS
#
# Embedding: transforma texto em um vetor de números que representa
# o significado semântico. Textos similares ficam próximos no espaço vetorial.
#
# Usamos sentence-transformers localmente — sem custo de API.

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modelo leve de embeddings em português/multilingual
print("Carregando modelo de embeddings...")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Texto que representa cada produto no índice
# Incluímos nome e categoria para a busca semântica funcionar bem
textos = [
    f"{p['nome']} | categoria: {p['categoria']} | preço: R$ {p['preco']:.2f}"
    for p in catalogo
]

print("Gerando embeddings...")
embeddings = embedder.encode(textos, convert_to_numpy=True)
embeddings = embeddings.astype("float32")

# Criar índice FAISS (busca por similaridade de cosseno via normalização L2)
faiss.normalize_L2(embeddings)
dimensao = embeddings.shape[1]
indice = faiss.IndexFlatIP(dimensao)  # Inner Product = cosseno após normalização
indice.add(embeddings)

print(f"Índice FAISS criado: {indice.ntotal} produtos indexados (dimensão {dimensao}).")

Carregando modelo de embeddings...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4102.52it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gerando embeddings...
Índice FAISS criado: 16 produtos indexados (dimensão 384).


Criamos `invocar()` como atalho para chamar o agente sem repetir o dicionário de mensagens a cada teste.

Testamos o fluxo completo: o agente recebe um pedido genérico, busca no catálogo com `search_products` e envia os resultados por e-mail com `send_email` — tudo em uma única ação composta.

Testamos com uma categoria diferente. A busca semântica encontra os produtos certos mesmo com termos genéricos que não aparecem literalmente no catálogo.

O modo verbose expõe o ciclo completo: `search_products` → resultado → composição do e-mail → `send_email`. É o padrão RAG em ação: **Retrieve → Augment → Generate**.

In [7]:
# Célula 7 — Testar a busca no índice diretamente (sem agente)
#
# Vamos fazer uma busca manual para entender o mecanismo antes de
# encapsulá-lo numa tool.

def buscar_produtos(query: str, k: int = 3) -> list[dict]:
    """Busca os k produtos mais relevantes para a query."""
    vetor = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(vetor)
    scores, indices = indice.search(vetor, k)
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        produto = catalogo[idx].copy()
        produto["score"] = float(score)
        resultados.append(produto)
    return resultados


# Teste 1: vestuário
print("=== Busca: 'roupas femininas' ===")
for p in buscar_produtos("roupas femininas", k=3):
    print(f"  score={p['score']:.3f} | {p['nome']:<35} R$ {p['preco']:.2f}")

print()

# Teste 2: eletrônicos
print("=== Busca: 'eletrônico tecnologia' ===")
for p in buscar_produtos("eletrônico tecnologia", k=3):
    print(f"  score={p['score']:.3f} | {p['nome']:<35} R$ {p['preco']:.2f}")

=== Busca: 'roupas femininas' ===
  score=0.739 | Calça social feminina               R$ 149.90
  score=0.670 | Blusa de algodão feminina           R$ 59.90
  score=0.606 | Blusa polo masculina                R$ 79.90

=== Busca: 'eletrônico tecnologia' ===
  score=0.319 | Smartphone 128GB                    R$ 1299.90
  score=0.221 | Brinquedo educativo blocos          R$ 89.90
  score=0.212 | Fone de ouvido bluetooth            R$ 199.90


---
## Parte B — Tool `search_products`

Agora encapsulamos a busca numa `@tool` para que o agente possa chamá-la.

In [8]:
# Célula 8 — Tool: search_products
from langchain_core.tools import tool
from typing import Optional

@tool
def search_products(query: str, k: int = 3) -> str:
    """Busca produtos no catálogo da loja por similaridade semântica.

    Use esta tool quando o usuário pedir produtos de uma categoria ou tipo específico.
    Retorna nome, categoria e preço dos produtos mais relevantes.

    Args:
        query: Descrição do que buscar (ex: 'roupas femininas', 'vestuário masculino', 'eletrônicos')
        k: Quantidade de produtos a retornar (padrão: 3)
    """
    resultados = buscar_produtos(query, k=k)
    if not resultados:
        return "Nenhum produto encontrado."

    linhas = [f"{len(resultados)} produto(s) encontrado(s) para '{query}':\n"]
    for p in resultados:
        linhas.append(
            f"- {p['nome']} | categoria: {p['categoria']} | R$ {p['preco']:.2f}"
        )
    return "\n".join(linhas)


# Teste direto da tool
print(search_products.invoke({"query": "vestuário", "k": 4}))

4 produto(s) encontrado(s) para 'vestuário':

- Jaqueta corta-vento | categoria: vestuario | R$ 249.90
- Vestido casual listrado | categoria: vestuario | R$ 119.90
- Blusa polo masculina | categoria: vestuario | R$ 79.90
- Calça social feminina | categoria: vestuario | R$ 149.90


Adicionamos a tool `send_email` — a mesma dos dias anteriores — para que o agente possa enviar os resultados da busca por e-mail.

In [9]:
# Célula 9 — Tool: send_email (mesma dos dias anteriores)

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Use esta tool quando o usuário pedir para enviar um e-mail.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: E-mails em cópia, separados por vírgula (opcional)
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc

    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

print("Tools disponíveis:", [search_products.name, send_email.name])

Tools disponíveis: ['search_products', 'send_email']


---
## Parte C — Agente RAG

O agente tem duas tools: `search_products` e `send_email`.  
Dado um pedido como *"manda um e-mail para aluno02 com 3 produtos de vestuário"*, ele deve:

1. Chamar `search_products("vestuário", k=3)`
2. Usar os resultados para montar o corpo do e-mail
3. Chamar `send_email(...)` com o conteúdo gerado

Isso é RAG na prática: **Retrieve** (buscar no FAISS) → **Augment** (incluir no prompt) → **Generate** (montar e enviar o e-mail).

Criamos `invocar()` como atalho para chamar o agente sem repetir o dicionário de mensagens a cada teste.

Testamos o fluxo completo: o agente recebe um pedido genérico, busca no catálogo e envia os resultados por e-mail em uma única ação composta.

Testamos com uma categoria diferente. A busca semântica encontra os produtos corretos mesmo sem correspondência exata de palavras.

O modo verbose expõe o ciclo completo: `search_products` → resultado → composição do e-mail → `send_email`. É o padrão RAG em ação: **Retrieve → Augment → Generate**.

In [10]:
# Célula 10 — Criar o agente RAG
from langchain.agents import create_agent

agente_rag = create_agent(
    model=llm,
    tools=[search_products, send_email],
    system_prompt=(
        "Você é um assistente de vendas. "
        "Quando o usuário pedir para enviar produtos por e-mail, use search_products "
        "para buscar os produtos relevantes e depois send_email para enviar a lista. "
        "Inclua nome e preço de cada produto no corpo do e-mail. "
        "Os endereços seguem o padrão alunoXX@curso.ia."
    ),
)

print("Agente RAG criado.")

Agente RAG criado.


In [11]:
# Célula 11 — Helper de invocação

def invocar(prompt, agente=agente_rag):
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("Helper pronto.")

Helper pronto.


In [12]:
# Célula 12 — Teste 1: buscar e enviar produtos de vestuário
resposta = invocar(
    "Manda um e-mail para aluno02@curso.ia com Ofertas de vestuário."   
)
print(resposta)

Pronto! ✅ E-mail enviado com sucesso para **aluno02@curso.ia** com 5 ofertas de vestuário, incluindo:
- Jaqueta corta-vento (R$ 249.90)
- Vestido casual listrado (R$ 119.90)
- Blusa polo masculina (R$ 79.90)
- Calça social feminina (R$ 149.90)
- Blusa de algodão feminina (R$ 59.90)


In [13]:
# Célula 13 — Teste 2: categoria diferente
resposta = invocar(
    "Envia para aluno03@curso.ia os 2 produtos de eletrônicos mais relevantes "
    
)
print(resposta)

Pronto! ✅ Enviei um e-mail para **aluno03@curso.ia** com os 2 produtos de eletrônicos mais relevantes:

1. **Smartphone 128GB** - R$ 1.299,90
2. **Fone de ouvido bluetooth** - R$ 199,90


In [14]:
# Célula 14 — Modo verbose: ver search → send em ação

def invocar_verbose(prompt, agente=agente_rag):
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 10},
    )
    for msg in resultado["messages"]:
        tipo = msg.__class__.__name__
        print(f"\n{'='*40}")
        print(f"  {tipo}")
        print(f"{'='*40}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → tool: {tc['name']}")
                print(f"     args: {tc['args']}")
        elif hasattr(msg, "content"):
            conteudo = str(msg.content)
            print(conteudo[:400] + ("..." if len(conteudo) > 400 else ""))

invocar_verbose(
    "Manda um e-mail para aluno04@curso.ia com 2 produtos de vestuário feminino "
    "e assunto 'Coleção feminina'."
)


  HumanMessage
Manda um e-mail para aluno04@curso.ia com 2 produtos de vestuário feminino e assunto 'Coleção feminina'.

  AIMessage
  → tool: search_products
     args: {'query': 'vestuário feminino', 'k': 2}

  ToolMessage
2 produto(s) encontrado(s) para 'vestuário feminino':

- Calça social feminina | categoria: vestuario | R$ 149.90
- Blusa de algodão feminina | categoria: vestuario | R$ 59.90

  AIMessage
  → tool: send_email
     args: {'to': 'aluno04@curso.ia', 'subject': 'Coleção feminina', 'body': 'Olá!\n\nConfira nossa coleção feminina com os seguintes produtos:\n\n1. Calça social feminina - R$ 149.90\n2. Blusa de algodão feminina - R$ 59.90\n\nFico à sua disposição para mais informações!\n\nAtenciosamente,\nAssistente de Vendas'}

  ToolMessage
E-mail enviado com sucesso para 'aluno04@curso.ia'.

  AIMessage
Perfeito! ✅ Enviei o e-mail para aluno04@curso.ia com o assunto "Coleção feminina" incluindo os 2 produtos de vestuário feminino:

1. **Calça social feminina** - R$ 149

---
## Resumo do Dia 3

| Conceito | O que aprendemos |
|---|---|
| Embedding | Transformar texto em vetor numérico que representa significado |
| FAISS | Índice vetorial local para busca por similaridade semântica |
| RAG | Retrieve → Augment → Generate: buscar antes de responder |
| `search_products` | Tool que encapsula a busca no FAISS |
| Agente RAG | Combina busca de conhecimento externo com envio de e-mail |

### O que vem no Dia 4

- Substituir `create_agent` por **LangGraph** — controle explícito do fluxo
- Nó condicional: se o assunto do e-mail contém "cobrança", o grafo bifurca
- **Human-in-the-loop**: o grafo pausa e pede confirmação antes de enviar